# EXP-004 — Loss neo text, λ = 1, seed 2 và 3
Cấu hình **đã chốt trước** từ seed 1: neo `cross`, ngân hàng A, λ = 1, lr 1e-3, 30 epoch, test ở epoch 5 / 10 / 20 / 30.
Seed 1 đã chạy (`EXP_004_anchor_train_seed1.ipynb`). Notebook này chạy seed 2 và 3, rồi tính trung bình 3 seed và so với **EXP-000 trung bình 3 seed** và paper.

# Cấu hình

In [1]:
SEEDS = [2, 3]
EPOCHS = 30
SAVE_EPOCHS = [5, 10, 20, 30]         # test head ở các mốc này
LR = 1e-3
LAMBDAS = [1]                         # trọng số loss neo (chốt từ seed 1)
ANCHOR = "cross"                      # cross: KL ảnh→text bank | text: MSE ma trận cos giữa các câu
BANK_VARIANT = "A"                    # A: không tên lớp | B: thêm từ cảm xúc (ablation)
job_of = lambda lam, seed: f"exp004_anchor_{ANCHOR}{BANK_VARIANT}_l{lam:g}_l14_seed{seed}"
BRANCH = "exp/004-text-anchor-loss"

INSTRUCTION = "Please play the role of a facial action describer. Objectively describe the detailed facial actions of the person in the image."
TRAIN_DATA = "/content/CAER-S_Face/train"
FACE_ZIP = "/content/drive/MyDrive/Datasets/CAERS_Face_train.zip"
TEST_ZIP = "/content/drive/MyDrive/Datasets/static_test_sets.zip"
PRE_DIR = "/content/drive/MyDrive/ExpCLIP_output/exp001/full"     # target đã cache ở EXP-001 (targets/chunk_*.npz)
CLIP_CACHE = f"{PRE_DIR}/clip_feats_l14.npz"                       # feature CLIP, tính 1 lần
OUT_DIR = "/content/drive/MyDrive/ExpCLIP_output/exp004"
BANK = f"{OUT_DIR}/text_bank_{BANK_VARIANT}_l14.npz"
SEED1_CSV = f"{OUT_DIR}/exp004_anchor_{ANCHOR}{BANK_VARIANT}_l14_seed1-results.csv"   # kết quả seed 1

import os, subprocess
def run(cmd):
    """Chạy lệnh shell, in output trực tiếp; lỗi thì dừng cell (khác với `!`)."""
    # bash + pipefail: lệnh có `| tee` vẫn báo lỗi nếu python lỗi
    p = subprocess.Popen("set -o pipefail; " + cmd, shell=True, executable="/bin/bash", text=True,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         env={**os.environ, "PYTHONUNBUFFERED": "1"})  # in log ngay, không đợi hết
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Lệnh lỗi (exit {p.returncode}) — xem output ở trên")

# Mount Drive & GPU

In [2]:
from google.colab import drive
drive.mount('/content/drive')
os.makedirs(OUT_DIR, exist_ok=True)
# không nạp BLIP-2 → GPU nào cũng được (T4/L4/A100)
!nvidia-smi --query-gpu=name,memory.total --format=csv

Mounted at /content/drive
name, memory.total [MiB]
NVIDIA A100-SXM4-40GB, 40960 MiB


# Clone repo (nhánh EXP-004)

In [3]:
if not os.path.isdir("/content/Exp-CLIP"):
    !git clone -b $BRANCH https://github.com/huongng389/Exp-CLIP.git /content/Exp-CLIP
%cd /content/Exp-CLIP
assert os.getcwd() == "/content/Exp-CLIP", "clone lỗi — nhánh đã push chưa?"
!git checkout $BRANCH && git pull
!git log --oneline -1   # ghi lại commit vào research_log

Cloning into '/content/Exp-CLIP'...
remote: Enumerating objects: 405, done.
remote: Counting objects: 100% (405/405), done.
remote: Compressing objects: 100% (135/135), done.
remote: Total 405 (delta 296), reused 374 (delta 266), pack-reused 0 (from 0)
Receiving objects: 100% (405/405), 32.23 MiB | 32.07 MiB/s, done.
Resolving deltas: 100% (296/296), done.
/content/Exp-CLIP
Already on 'exp/004-text-anchor-loss'
Your branch is up to date with 'origin/exp/004-text-anchor-loss'.
Already up to date.
76f4644 (HEAD -> exp/004-text-anchor-loss, origin/exp/004-text-anchor-loss) EXP-004: add executed anchor-loss notebook (seed 1, lambda 0.1/1/10)


# Giải nén dữ liệu (train + test)

In [4]:
!mkdir -p /content/CAER-S_Face /content/static_test_sets
!unzip -q -n "$FACE_ZIP" -d /content/CAER-S_Face
!unzip -q -n "$TEST_ZIP" -d /content/static_test_sets
!ls /content/CAER-S_Face/train
!ls /content/static_test_sets

Angry  Disgust	Fear  Happy  Neutral  Sad  Surprise
AffectNet7  AffectNet8	FERPlus  RAFDB


# Cài dependencies (venv310)\ntorch 2.0.1 + lavis: dùng cho target, proto, weight, train, test.

In [5]:
!sudo apt-get -qq update > /dev/null
!sudo apt-get -qq install -y python3.10 python3.10-venv python3.10-distutils > /dev/null
!python3.10 -m venv /content/venv310
!/content/venv310/bin/pip install -q --upgrade pip
!/content/venv310/bin/pip install -q --no-cache-dir torch==2.0.1 torchvision==0.15.2 torchaudio==2.0.2 --index-url https://download.pytorch.org/whl/cu118
!/content/venv310/bin/pip install -q --no-cache-dir transformers tqdm regex pyyaml pandas pydantic click timm einops
!/content/venv310/bin/pip install -q --no-cache-dir salesforce-lavis
# phải cài SAU lavis: lavis kéo numpy 2.x / opencv mới
!/content/venv310/bin/pip install -q --no-cache-dir "numpy<2" "opencv-python==4.10.0.84" "opencv-python-headless==4.10.0.84"
!/content/venv310/bin/pip install -q --no-cache-dir matplotlib scikit-learn
!/content/venv310/bin/python -c "import numpy, torch, lavis, sklearn; print('numpy', numpy.__version__, '| torch', torch.__version__, '| cuda', torch.cuda.is_available())"

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 79, <STDIN> line 7.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 22.6 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Instal

# Cache feature CLIP + ngân hàng text
Đã có trên Drive từ seed 1 thì bỏ qua.

In [6]:
if not os.path.exists(CLIP_CACHE):
    run(f'/content/venv310/bin/python tools/cache_clip_features.py --train-data {TRAIN_DATA} --out {CLIP_CACHE} '
        f'--load-model CLIP_L14 --batch-size 256 --workers 8')
if not os.path.exists(BANK):
    run(f'/content/venv310/bin/python tools/build_text_bank.py --variant {BANK_VARIANT} --out {BANK} --load-model CLIP_L14')
!ls -lh "$CLIP_CACHE" "$BANK"

-rw------- 1 root root  75M Oct 14 07:10 /content/drive/MyDrive/ExpCLIP_output/exp001/full/clip_feats_l14.npz
-rw------- 1 root root 576K Oct 14 15:16 /content/drive/MyDrive/ExpCLIP_output/exp004/text_bank_A_l14.npz


# Train (seed 2, 3; λ = 1; 30 epoch, từ cache)
Bị ngắt thì chạy lại cell này: phần đã xong được bỏ qua, phần đang dở tự resume.

In [7]:
for seed in SEEDS:
  for lam in LAMBDAS:
    job = job_of(lam, seed)
    if os.path.exists(f"{OUT_DIR}/{job}-e{SAVE_EPOCHS[-1]}-model.pth"):
        print(f"seed {seed}, λ={lam}: đã train xong, bỏ qua"); continue
    resume = f"{OUT_DIR}/{job}-resume.pth"
    print(f"=============== seed {seed}, λ = {lam}")
    run(f'/content/venv310/bin/python train.py '
        f'--epochs {EPOCHS} --batch-size 512 --batch-size-test-image 512 --batch-size-test-video 64 '
        f'--lr {LR} --weight-decay 1e-4 --momentum 0.9 --print-freq 1000 --milestones 1000 --seed {seed} '
        f'--job-id {job} --instruction "{INSTRUCTION}" --load-model CLIP_L14 '
        f'--cache-clip {CLIP_CACHE} --cache-targets {PRE_DIR}/targets '
        f'--text-bank {BANK} --anchor-weight {lam} --anchor-type {ANCHOR} '
        f'--save-epochs {" ".join(map(str, SAVE_EPOCHS))} --extra-checkpoint-dir {OUT_DIR}'
        + (f' --resume {resume}' if os.path.exists(resume) else ''))

=============== seed 2, λ = 1
Training date:  26-10-15 10:11
************************
workers = 8
epochs = 30
batch_size = 512
batch_size_test_image = 512
batch_size_test_video = 64
lr = 0.001
weight_decay = 0.0001
momentum = 0.9
print_freq = 1000
milestones = [1000]
seed = 2
job_id = exp004_anchor_crossA_l1_l14_seed2
instruction = Please play the role of a facial action describer. Objectively describe the detailed facial actions of the person in the image.
load_model = CLIP_L14
extra_checkpoint_dir = /content/drive/MyDrive/ExpCLIP_output/exp004
train_data = /data/EECS-IoannisLab/datasets/Static_FER_Datasets/CAERS_Face/train
sample_weights = None
cache_clip = /content/drive/MyDrive/ExpCLIP_output/exp001/full/clip_feats_l14.npz
cache_targets = /content/drive/MyDrive/ExpCLIP_output/exp001/full/targets
text_bank = /content/drive/MyDrive/ExpCLIP_output/exp004/text_bank_A_l14.npz
anchor_weight = 1.0
anchor_type = cross
save_epochs = [5, 10, 20, 30]
resume = None
************************

  

# Test zero-shot từng mốc

In [8]:
!mkdir -p checkpoint log
for seed in SEEDS:
  for lam in LAMBDAS:
    for e in SAVE_EPOCHS:
        job = f"{job_of(lam, seed)}-e{e}"
        if os.path.exists(f"{OUT_DIR}/{job}-test.txt"):
            !cp "$OUT_DIR/$job-test.txt" log/
            continue
        if not os.path.exists(f"checkpoint/{job}-model.pth"):
            !cp "$OUT_DIR/$job-model.pth" checkpoint/
        print(f"=============== seed {seed}, λ = {lam}, epoch {e}")
        run(f'/content/venv310/bin/python test.py --load-model CLIP_L14 --job-id {job} '
            f'--data-root /content/static_test_sets 2>&1 | tee log/{job}-test.txt')
        !cp log/$job-test.txt "$OUT_DIR/"

=============== seed 2, λ = 1, epoch 5
************************
load_model = CLIP_L14
job_id = exp004_anchor_crossA_l1_l14_seed2-e5
data_root = /content/static_test_sets
************************
************************************************************************** Zero-shot Prompt Type:  Class Name
******************** Static FER Zero-shot Performance ********************
************************* RAFDB
UAR/WAR: 49.78/51.66
************************* AffectNet7
UAR/WAR: 39.69/39.69
************************* AffectNet8
UAR/WAR: 34.05/34.05
************************* FERPlus
UAR/WAR: 37.03/40.89
************************************************************************** Zero-shot Prompt Type:  An Expression of Name
******************** Static FER Zero-shot Performance ********************
************************* RAFDB
UAR/WAR: 52.61/58.57
************************* AffectNet7
UAR/WAR: 42.71/42.71
************************* AffectNet8
UAR/WAR: 38.00/38.00
************************* FERPl

# So sánh
Bảng 1: Ensemble Five từng seed. Bảng 2: trung bình ± std 3 seed, Δ so với EXP-000 trung bình 3 seed và paper. Bảng 3: đủ 5 prompt, trung bình 3 seed, Δ so với EXP-000 trung bình 3 seed.

In [9]:
import re, pandas as pd
REF = pd.DataFrame([{"prompt": "Class Name", "dataset": "RAFDB", "src": "EXP-000 seed1", "UAR": 50.36, "WAR": 56.58}, {"prompt": "Class Name", "dataset": "AffectNet7", "src": "EXP-000 seed1", "UAR": 42.51, "WAR": 42.51}, {"prompt": "Class Name", "dataset": "AffectNet8", "src": "EXP-000 seed1", "UAR": 36.95, "WAR": 36.95}, {"prompt": "Class Name", "dataset": "FERPlus", "src": "EXP-000 seed1", "UAR": 38.27, "WAR": 45.45}, {"prompt": "An Expression of Name", "dataset": "RAFDB", "src": "EXP-000 seed1", "UAR": 56.73, "WAR": 64.05}, {"prompt": "An Expression of Name", "dataset": "AffectNet7", "src": "EXP-000 seed1", "UAR": 44.0, "WAR": 44.0}, {"prompt": "An Expression of Name", "dataset": "AffectNet8", "src": "EXP-000 seed1", "UAR": 38.33, "WAR": 38.33}, {"prompt": "An Expression of Name", "dataset": "FERPlus", "src": "EXP-000 seed1", "UAR": 39.99, "WAR": 52.92}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "RAFDB", "src": "EXP-000 seed1", "UAR": 54.94, "WAR": 65.12}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet7", "src": "EXP-000 seed1", "UAR": 43.97, "WAR": 43.97}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet8", "src": "EXP-000 seed1", "UAR": 38.02, "WAR": 38.02}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "FERPlus", "src": "EXP-000 seed1", "UAR": 36.63, "WAR": 48.89}, {"prompt": "Expression Ensemble Five", "dataset": "RAFDB", "src": "EXP-000 seed1", "UAR": 58.88, "WAR": 66.88}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet7", "src": "EXP-000 seed1", "UAR": 44.09, "WAR": 44.09}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet8", "src": "EXP-000 seed1", "UAR": 39.23, "WAR": 39.23}, {"prompt": "Expression Ensemble Five", "dataset": "FERPlus", "src": "EXP-000 seed1", "UAR": 40.65, "WAR": 51.47}, {"prompt": "Expression Ensemble Ten", "dataset": "RAFDB", "src": "EXP-000 seed1", "UAR": 57.73, "WAR": 65.91}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet7", "src": "EXP-000 seed1", "UAR": 44.94, "WAR": 44.94}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet8", "src": "EXP-000 seed1", "UAR": 39.55, "WAR": 39.55}, {"prompt": "Expression Ensemble Ten", "dataset": "FERPlus", "src": "EXP-000 seed1", "UAR": 40.45, "WAR": 49.29}, {"prompt": "Class Name", "dataset": "RAFDB", "src": "EXP-000 mean3", "UAR": 50.23, "WAR": 54.56}, {"prompt": "Class Name", "dataset": "AffectNet7", "src": "EXP-000 mean3", "UAR": 41.47, "WAR": 41.47}, {"prompt": "Class Name", "dataset": "AffectNet8", "src": "EXP-000 mean3", "UAR": 35.76, "WAR": 35.76}, {"prompt": "Class Name", "dataset": "FERPlus", "src": "EXP-000 mean3", "UAR": 38.58, "WAR": 45.82}, {"prompt": "An Expression of Name", "dataset": "RAFDB", "src": "EXP-000 mean3", "UAR": 55.34, "WAR": 62.0}, {"prompt": "An Expression of Name", "dataset": "AffectNet7", "src": "EXP-000 mean3", "UAR": 43.79, "WAR": 43.79}, {"prompt": "An Expression of Name", "dataset": "AffectNet8", "src": "EXP-000 mean3", "UAR": 38.4, "WAR": 38.4}, {"prompt": "An Expression of Name", "dataset": "FERPlus", "src": "EXP-000 mean3", "UAR": 38.89, "WAR": 51.62}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "RAFDB", "src": "EXP-000 mean3", "UAR": 55.64, "WAR": 64.78}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet7", "src": "EXP-000 mean3", "UAR": 44.39, "WAR": 44.39}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet8", "src": "EXP-000 mean3", "UAR": 38.08, "WAR": 38.08}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "FERPlus", "src": "EXP-000 mean3", "UAR": 37.95, "WAR": 49.17}, {"prompt": "Expression Ensemble Five", "dataset": "RAFDB", "src": "EXP-000 mean3", "UAR": 58.07, "WAR": 65.32}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet7", "src": "EXP-000 mean3", "UAR": 44.17, "WAR": 44.17}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet8", "src": "EXP-000 mean3", "UAR": 38.84, "WAR": 38.84}, {"prompt": "Expression Ensemble Five", "dataset": "FERPlus", "src": "EXP-000 mean3", "UAR": 41.55, "WAR": 53.41}, {"prompt": "Expression Ensemble Ten", "dataset": "RAFDB", "src": "EXP-000 mean3", "UAR": 57.19, "WAR": 64.6}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet7", "src": "EXP-000 mean3", "UAR": 44.97, "WAR": 44.97}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet8", "src": "EXP-000 mean3", "UAR": 39.45, "WAR": 39.45}, {"prompt": "Expression Ensemble Ten", "dataset": "FERPlus", "src": "EXP-000 mean3", "UAR": 42.17, "WAR": 49.46}, {"prompt": "Class Name", "dataset": "RAFDB", "src": "paper", "UAR": 58.7, "WAR": 65.37}, {"prompt": "Class Name", "dataset": "AffectNet7", "src": "paper", "UAR": 44.27, "WAR": 44.27}, {"prompt": "Class Name", "dataset": "AffectNet8", "src": "paper", "UAR": 38.44, "WAR": 38.43}, {"prompt": "Class Name", "dataset": "FERPlus", "src": "paper", "UAR": 48.28, "WAR": 55.42}, {"prompt": "An Expression of Name", "dataset": "RAFDB", "src": "paper", "UAR": 58.7, "WAR": 65.37}, {"prompt": "An Expression of Name", "dataset": "AffectNet7", "src": "paper", "UAR": 44.27, "WAR": 44.27}, {"prompt": "An Expression of Name", "dataset": "AffectNet8", "src": "paper", "UAR": 38.44, "WAR": 38.43}, {"prompt": "An Expression of Name", "dataset": "FERPlus", "src": "paper", "UAR": 48.28, "WAR": 55.42}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "RAFDB", "src": "paper", "UAR": 58.7, "WAR": 65.37}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet7", "src": "paper", "UAR": 44.27, "WAR": 44.27}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "AffectNet8", "src": "paper", "UAR": 38.44, "WAR": 38.43}, {"prompt": "A Photo of A Face with An Expression of Name", "dataset": "FERPlus", "src": "paper", "UAR": 48.28, "WAR": 55.42}, {"prompt": "Expression Ensemble Five", "dataset": "RAFDB", "src": "paper", "UAR": 58.7, "WAR": 65.37}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet7", "src": "paper", "UAR": 44.27, "WAR": 44.27}, {"prompt": "Expression Ensemble Five", "dataset": "AffectNet8", "src": "paper", "UAR": 38.44, "WAR": 38.43}, {"prompt": "Expression Ensemble Five", "dataset": "FERPlus", "src": "paper", "UAR": 48.28, "WAR": 55.42}, {"prompt": "Expression Ensemble Ten", "dataset": "RAFDB", "src": "paper", "UAR": 58.7, "WAR": 65.37}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet7", "src": "paper", "UAR": 44.27, "WAR": 44.27}, {"prompt": "Expression Ensemble Ten", "dataset": "AffectNet8", "src": "paper", "UAR": 38.44, "WAR": 38.43}, {"prompt": "Expression Ensemble Ten", "dataset": "FERPlus", "src": "paper", "UAR": 48.28, "WAR": 55.42}])

def parse(path):
    rows, prompt, dataset = [], None, None
    for line in open(path):
        if "Prompt Type:" in line:
            prompt = line.split("Prompt Type:")[1].strip()
        elif re.match(r"^\*+ (RAFDB|AffectNet7|AffectNet8|FERPlus)$", line.strip()):
            dataset = line.strip().split()[-1]
        elif line.startswith("UAR/WAR:"):
            uar, war = map(float, line.split(":")[1].strip().split("/"))
            rows.append(dict(prompt=prompt, dataset=dataset, UAR=uar, WAR=war))
    return pd.DataFrame(rows)

res = pd.concat([parse(f"log/{job_of(lam, seed)}-e{e}-test.txt").assign(seed=seed, lam=lam, epoch=e)
                 for seed in SEEDS for lam in LAMBDAS for e in SAVE_EPOCHS])
res.to_csv(f"{OUT_DIR}/exp004_anchor_{ANCHOR}{BANK_VARIANT}_l14_seed{'_'.join(map(str, SEEDS))}-results.csv", index=False)
r1 = pd.read_csv(SEED1_CSV).assign(seed=1)
res = pd.concat([r1[r1.lam.isin(LAMBDAS)], res])
assert res.groupby(["lam", "epoch"]).seed.nunique().eq(3).all(), "thiếu seed"
pd.set_option("display.width", 250)
DS = ["RAFDB", "AffectNet7", "AffectNet8", "FERPlus"]
E5 = "Expression Ensemble Five"
m3 = REF[REF.src == "EXP-000 mean3"].set_index(["prompt", "dataset"]).UAR
paper = REF[(REF.src == "paper") & (REF.prompt == E5)].set_index("dataset").UAR

ens = res[res.prompt == E5]
display(ens.pivot_table(index=["lam", "epoch"], columns=["dataset", "seed"], values="UAR")[DS])

g = ens.groupby(["lam", "epoch", "dataset"]).UAR
mean, std = g.mean().unstack()[DS], g.std().unstack()[DS]
display(pd.concat({"mean": mean.round(2), "std": std.round(2),
                   "Δ vs EXP-000 mean3": (mean - m3.loc[E5]).round(2),
                   "Δ vs paper": (mean - paper).round(2)}, axis=1))

full = res.groupby(["lam", "epoch", "prompt", "dataset"]).UAR.mean().reset_index()
full = full.merge(m3.rename("UAR_base").reset_index(), on=["prompt", "dataset"])
full["Δ"] = (full.UAR - full.UAR_base).round(2)
display(full.pivot_table(index=["prompt", "dataset"], columns=["lam", "epoch"], values="Δ", sort=False))

dataset    RAFDB               AffectNet7               AffectNet8               FERPlus              
seed           1      2      3          1      2      3          1      2      3       1      2      3
lam epoch                                                                                             
1.0 5      59.39  56.69  53.35      44.09  42.54  43.14      38.70  37.85  37.33   43.27  41.46  38.56
    10     60.72  57.69  56.08      44.31  43.51  44.29      38.88  38.42  38.02   44.05  41.97  41.32
    20     59.74  57.28  57.80      45.14  43.97  45.20      39.15  38.65  38.65   45.05  43.49  44.35
    30     58.73  57.07  58.26      45.83  43.97  45.46      39.60  38.75  39.07   46.14  45.63  45.34

mean                                 std                               Δ vs EXP-000 mean3                               Δ vs paper                              
dataset    RAFDB AffectNet7 AffectNet8 FERPlus RAFDB AffectNet7 AffectNet8 FERPlus              RAFDB AffectNet7 AffectNet8 FERPlus      RAFDB AffectNet7 AffectNet8 FERPlus
lam epoch                                                                                                                                                                   
1.0 5      56.48      43.26      37.96   41.10  3.03       0.78       0.69    2.38              -1.59      -0.91      -0.88   -0.45      -2.22      -1.01      -0.48   -7.18
    10     58.16      44.04      38.44   42.45  2.36       0.46       0.43    1.43               0.09      -0.13      -0.40    0.90      -0.54      -0.23       0.00   -5.83
    20     58.27      44.77      38.82   44.30  1.30       0.69       0.29    0.78               0.20       0.60      -0.02    2.75      -0.43       0.50       0.38   -3.98
    30     58.02      45.09      39.14   45.70  0.86       0.98       0.43    0.41              -0.05       0.92       0.30    4.15      -0.68       0.82       0.70   -2.58

lam                                                       1.0                  
epoch                                                      5     10    20    30
prompt                                       dataset                           
A Photo of A Face with An Expression of Name AffectNet7 -2.70 -1.52 -0.76 -0.37
                                             AffectNet8 -2.89 -1.94 -1.27 -0.79
                                             FERPlus    -0.86  2.18  5.49  5.93
                                             RAFDB      -0.23  1.56  3.47  3.13
An Expression of Name                        AffectNet7 -0.57 -0.48 -0.12  0.54
                                             AffectNet8  0.09 -0.10  0.25  0.75
                                             FERPlus    -0.18  2.15  5.63  6.79
                                             RAFDB      -1.36  0.31  1.39  1.52
Class Name                                   AffectNet7 -1.56 -0.74 -0.01  0.37
                                             AffectNet8 -1.53 -0.77 -0.08  0.32
                                             FERPlus     1.43  1.86  4.38  4.39
                                             RAFDB      -0.46  1.95  3.73  3.38
Expression Ensemble Five                     AffectNet7 -0.91 -0.13  0.60  0.92
                                             AffectNet8 -0.88 -0.40 -0.02  0.30
                                             FERPlus    -0.45  0.90  2.75  4.15
                                             RAFDB      -1.59  0.09  0.20 -0.05
Expression Ensemble Ten                      AffectNet7 -0.76 -0.33  0.01  0.12
                                             AffectNet8 -0.96 -0.73 -0.52 -0.30
                                             FERPlus    -0.39  1.51  2.88  3.90
                                             RAFDB      -0.15  0.85  1.32  1.25

# Log train
Accuracy train và loss neo trung bình ở mỗi mốc epoch.

In [10]:
import re
for seed in SEEDS:
  for lam in LAMBDAS:
    lines = open(f"{OUT_DIR}/{job_of(lam, seed)}-log.txt").read().splitlines()
    acc = [float(l.split(":")[1]) for l in lines if l.startswith("The best accuracy")]
    anc = [float(re.search(r"Anchor [\d.]+ \(([\d.]+)\)", l).group(1)) for l in lines if "Anchor" in l]
    print(f"seed {seed}, λ={lam}: acc", {e: round(acc[e - 1], 2) for e in SAVE_EPOCHS if e <= len(acc)},
          "| anchor", [round(a, 3) for a in anc[::max(1, len(anc) // 6)]])

seed 2, λ=1: acc {5: 40.41, 10: 60.68, 20: 75.29, 30: 81.5} | anchor [0.297, 0.084, 0.06, 0.055, 0.052, 0.049]
seed 3, λ=1: acc {5: 39.61, 10: 60.55, 20: 75.34, 30: 81.33} | anchor [0.322, 0.087, 0.062, 0.053, 0.052, 0.051]
